In [22]:
import json
import hashlib
import base64
import copy
import time
import secrets

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature

In [23]:
device_id  = "IOMT-DEV-00423"
patient_id = "PAT-9981"
nonce      = secrets.token_hex(8)
timestamp  = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

vitals = {
    "heart_rate":     82,
    "spo2":           97,
    "temperature":    36.7,
    "blood_pressure": "120/80"
}

packet = {
    "device_id":  device_id,
    "patient_id": patient_id,
    "timestamp":  timestamp,
    "nonce":      nonce,
    "vitals":     vitals
}

canonical = json.dumps(packet, sort_keys=True, separators=(',', ':')).encode('utf-8')
packet["hash_value"] = hashlib.sha256(canonical).hexdigest()



received_packet = packet

print(" Packet received :")
print(json.dumps(received_packet, indent=2))

 Packet received :
{
  "device_id": "IOMT-DEV-00423",
  "patient_id": "PAT-9981",
  "timestamp": "2026-05-03T15:11:28Z",
  "nonce": "12e11920215b30c3",
  "vitals": {
    "heart_rate": 82,
    "spo2": 97,
    "temperature": 36.7,
    "blood_pressure": "120/80"
  },
  "hash_value": "d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275"
}


In [24]:
def validate_received_packet(packet: dict) -> bool:
    # Fields your friend's code always produces
    required_fields = {"device_id", "patient_id", "timestamp", "nonce", "vitals", "hash_value"}
    missing = required_fields - set(packet.keys())

    if missing:
        print(f" Packet is missing fields: {missing}")
        return False

    print(" All fields present:")
    for k, v in packet.items():
        display = str(v)[:50] + "..." if len(str(v)) > 50 else str(v)
        print(f"   {k:<15}: {display}")
    return True

validate_received_packet(received_packet)

 All fields present:
   device_id      : IOMT-DEV-00423
   patient_id     : PAT-9981
   timestamp      : 2026-05-03T15:11:28Z
   nonce          : 12e11920215b30c3
   vitals         : {'heart_rate': 82, 'spo2': 97, 'temperature': 36.7...
   hash_value     : d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992...


True

In [25]:
def verify_friends_hash(packet: dict) -> bool:
    # Rebuild packet as it was BEFORE hash_value was added
    pre_hash_packet = {
        "device_id":  packet["device_id"],
        "patient_id": packet["patient_id"],
        "timestamp":  packet["timestamp"],
        "nonce":      packet["nonce"],
        "vitals":     packet["vitals"],
    }
    canonical       = json.dumps(pre_hash_packet, sort_keys=True, separators=(',', ':')).encode('utf-8')
    expected_hash   = hashlib.sha256(canonical).hexdigest()
    received_hash   = packet["hash_value"]

    print(f"   Expected hash_value : {expected_hash}")
    print(f"   Received hash_value : {received_hash}")

    if expected_hash == received_hash:
        print("\n hash_value is CORRECT —  packet is untampered")
        return True
    else:
        print("\n hash_value MISMATCH — packet may have been modified")
        return False

print(" Checking friend's hash_value...\n")
verify_friends_hash(received_packet)

 Checking friend's hash_value...

   Expected hash_value : d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275
   Received hash_value : d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275

 hash_value is CORRECT —  packet is untampered


True

In [26]:
private_key = ec.generate_private_key(ec.SECP256R1())
public_key  = private_key.public_key()

pub_pem = public_key.public_bytes(
    serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

print(" ECDSA Key Pair Generated (SECP256R1 / P-256)")
print("\n PUBLIC KEY — ")
print(pub_pem)

 ECDSA Key Pair Generated (SECP256R1 / P-256)

 PUBLIC KEY — 
-----BEGIN PUBLIC KEY-----
MFkwEwYHKoZIzj0CAQYIKoZIzj0DAQcDQgAEGnsJXeqS0dTqFiktytsqiSqW4k5S
9KUwq/D3RDyZYcRhvg73jGj865fv1tw5V9kM4cpvBwxedSeV+/eFYMZ2tw==
-----END PUBLIC KEY-----



In [27]:
def sign_packet(packet: dict, private_key) -> dict:
    """
    Signs the complete packet.
    Includes all fields: device_id, patient_id, timestamp,
    nonce, vitals, hash_value.
    """
    signed_packet = copy.deepcopy(packet)

    # Step 1 — canonical JSON of the full packet (no signature field yet)
    data_to_sign = {
        "device_id":  packet["device_id"],
        "patient_id": packet["patient_id"],
        "timestamp":  packet["timestamp"],
        "nonce":      packet["nonce"],
        "vitals":     packet["vitals"],
        "hash_value": packet["hash_value"],   # friend's hash is part of what you sign
    }
    canonical = json.dumps(data_to_sign, sort_keys=True, separators=(',', ':')).encode('utf-8')
    print(f"   Canonical JSON : {canonical[:70].decode()}...")

    # Step 2 — SHA-256 digest
    digest = hashlib.sha256(canonical).digest()
    print(f"   SHA-256 digest : {digest.hex()[:32]}...")

    # Step 3 — ECDSA sign
    signature_bytes = private_key.sign(digest, ec.ECDSA(hashes.SHA256()))

    # Step 4 — attach base64 signature
    signed_packet["signature"] = base64.b64encode(signature_bytes).decode()
    print(f"   Signature      : {signed_packet['signature'][:40]}...")

    return signed_packet


print(" Signing packet...\n")
signed_packet = sign_packet(received_packet, private_key)

print("\n Signing complete!")
print("\n Final signed packet:")
print(json.dumps(signed_packet, indent=2))

 Signing packet...

   Canonical JSON : {"device_id":"IOMT-DEV-00423","hash_value":"d9e87b2f81fadac8a892e7841f...
   SHA-256 digest : 09438836efc4f524bcabb1d055702ab8...
   Signature      : MEYCIQDid9eobgyiVUAHnmuQTYCGleXAPNhZ3bSx...

 Signing complete!

 Final signed packet:
{
  "device_id": "IOMT-DEV-00423",
  "patient_id": "PAT-9981",
  "timestamp": "2026-05-03T15:11:28Z",
  "nonce": "12e11920215b30c3",
  "vitals": {
    "heart_rate": 82,
    "spo2": 97,
    "temperature": 36.7,
    "blood_pressure": "120/80"
  },
  "hash_value": "d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275",
  "signature": "MEYCIQDid9eobgyiVUAHnmuQTYCGleXAPNhZ3bSx5ZMHjKrxXgIhAIdTNU7sK3LrMN+boWxd7268kktVComYX/ovtTeMHbBV"
}


In [28]:
def verify_signature(packet: dict, public_key) -> bool:
    """
    Verifies the ECDSA signature on the signed packet.
    Recomputes hash from the same fields used during signing.
    """
    print("─" * 55)

    # Step 1 — check signature field
    if "signature" not in packet:
        print(" FAIL: 'signature' field is missing")
        return False
    print("   [1/3] Signature field found   ")

    # Step 2 — recompute hash (same fields, same order as signing)
    data_to_verify = {
        "device_id":  packet["device_id"],
        "patient_id": packet["patient_id"],
        "timestamp":  packet["timestamp"],
        "nonce":      packet["nonce"],
        "vitals":     packet["vitals"],
        "hash_value": packet["hash_value"],
    }
    canonical = json.dumps(data_to_verify, sort_keys=True, separators=(',', ':')).encode('utf-8')
    digest    = hashlib.sha256(canonical).digest()
    print(f"   [2/3] Hash recomputed              ({digest.hex()[:20]}...)")

    # Step 3 — decode signature
    try:
        signature_bytes = base64.b64decode(packet["signature"])
        print(f"   [3/3] Signature decoded ({len(signature_bytes)} bytes)   ")
    except Exception as e:
        print(f" FAIL: Cannot decode signature — {e}")
        return False

    # Step 4 — ECDSA verify
    try:
        public_key.verify(signature_bytes, digest, ec.ECDSA(hashes.SHA256()))
        print("─" * 55)
        print(" SIGNATURE VALID — Packet is authentic")
        return True
    except InvalidSignature:
        print("─" * 55)
        print(" SIGNATURE INVALID — Packet was tampered!")
        return False


print(" Verifying signature...\n")
result = verify_signature(signed_packet, public_key)
print(f"\nFinal result: {'AUTHENTIC ' if result else 'REJECTED '}")

 Verifying signature...

───────────────────────────────────────────────────────
   [1/3] Signature field found   
   [2/3] Hash recomputed              (09438836efc4f524bcab...)
   [3/3] Signature decoded (72 bytes)   
───────────────────────────────────────────────────────
 SIGNATURE VALID — Packet is authentic

Final result: AUTHENTIC 


In [29]:
print("=" * 55)
print("TEST 1: Attacker changes heart_rate after signing")
print("=" * 55)
t1 = copy.deepcopy(signed_packet)
t1["vitals"]["heart_rate"] = 999
verify_signature(t1, public_key)

print()
print("=" * 55)
print("TEST 2: Attacker changes patient_id after signing")
print("=" * 55)
t2 = copy.deepcopy(signed_packet)
t2["patient_id"] = "PAT-0000"
verify_signature(t2, public_key)

print()
print("=" * 55)
print("TEST 3: Attacker tampers with hash_value")
print("=" * 55)
t3 = copy.deepcopy(signed_packet)
t3["hash_value"] = "aaaaaaaabbbbbbbbccccccccdddddddd" * 2
verify_signature(t3, public_key)

TEST 1: Attacker changes heart_rate after signing
───────────────────────────────────────────────────────
   [1/3] Signature field found   
   [2/3] Hash recomputed              (c29855149d74aa1a7df8...)
   [3/3] Signature decoded (72 bytes)   
───────────────────────────────────────────────────────
 SIGNATURE INVALID — Packet was tampered!

TEST 2: Attacker changes patient_id after signing
───────────────────────────────────────────────────────
   [1/3] Signature field found   
   [2/3] Hash recomputed              (200c7549a97029d4099a...)
   [3/3] Signature decoded (72 bytes)   
───────────────────────────────────────────────────────
 SIGNATURE INVALID — Packet was tampered!

TEST 3: Attacker tampers with hash_value
───────────────────────────────────────────────────────
   [1/3] Signature field found   
   [2/3] Hash recomputed              (442ed98d424c7d3b4be0...)
   [3/3] Signature decoded (72 bytes)   
───────────────────────────────────────────────────────
 SIGNATURE INVALID —

False

In [30]:
print("=" * 60)
print("  FULL PIPELINE SUMMARY")
print("=" * 60)

print("\n[FRIEND] Built packet with fields:")
print(f"   device_id, patient_id, timestamp, nonce, vitals, hash_value")
print(f"   hash_value = SHA-256 of canonical packet JSON")

print("\n[YOU — Step 1] Verified friend's hash_value")
h_ok = verify_friends_hash(received_packet)

print("\n[YOU — Step 2] Signed the full packet with ECDSA")
sp = sign_packet(received_packet, private_key)
print(f"   Added 'signature' field to packet")

print("\n[YOU — Step 3] Verified the ECDSA signature")
v_ok = verify_signature(sp, public_key)

print("\n" + "=" * 60)
print(f"  hash_value check : {'PASS ' if h_ok else 'FAIL '}")
print(f"  ECDSA verify     : {'PASS ' if v_ok else 'FAIL '}")
print(f"  OVERALL RESULT   : {'AUTHENTIC ' if (h_ok and v_ok) else 'REJECTED '}")
print("=" * 60)

  FULL PIPELINE SUMMARY

[FRIEND] Built packet with fields:
   device_id, patient_id, timestamp, nonce, vitals, hash_value
   hash_value = SHA-256 of canonical packet JSON

[YOU — Step 1] Verified friend's hash_value
   Expected hash_value : d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275
   Received hash_value : d9e87b2f81fadac8a892e7841f4c76bcefdf784a4a4f786992b7b86016da1275

 hash_value is CORRECT —  packet is untampered

[YOU — Step 2] Signed the full packet with ECDSA
   Canonical JSON : {"device_id":"IOMT-DEV-00423","hash_value":"d9e87b2f81fadac8a892e7841f...
   SHA-256 digest : 09438836efc4f524bcabb1d055702ab8...
   Signature      : MEUCIQDl3VARENOjcOS9QurzvswATwuoFieqsOxG...
   Added 'signature' field to packet

[YOU — Step 3] Verified the ECDSA signature
───────────────────────────────────────────────────────
   [1/3] Signature field found   
   [2/3] Hash recomputed              (09438836efc4f524bcab...)
   [3/3] Signature decoded (71 bytes)   
──────────────

In [31]:
with open("my_private_key.pem", "wb") as f:
    f.write(private_key.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.PKCS8,
        serialization.NoEncryption()
    ))

with open("my_public_key.pem", "wb") as f:
    f.write(public_key.public_bytes(
        serialization.Encoding.PEM,
        serialization.PublicFormat.SubjectPublicKeyInfo
    ))

print(" my_private_key.pem — keep this secret, never share")
print(" my_public_key.pem  — share this with teammates for verification")

 my_private_key.pem — keep this secret, never share
 my_public_key.pem  — share this with teammates for verification
